# GeometricNearestNeighbors usage examples

Anton Antonov   
June 2026

---

## Introduction

This notebook has the most typical workflow of using the Python package "GeometricNearestNeighbors", [AAp1].

---

## Setup

In [ ]:
from GeometricNearestNeighborsProcessor import *
from RandomDataGenerators import *
from OutlierIdentifiers import *

import numpy
import random

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

---

## Usage examples

Generate random points:

In [23]:
dfPoints = random_data_frame(n_rows=30, columns_spec = ["X", "Y"], generators= {"X": numpy.random.normal, "Y": numpy.random.normal})
print(dfPoints.shape)
print(dfPoints[1:6])

(30, 2)
          X         Y
1  1.972754 -1.353810
2  0.121834  0.273519
3 -0.022814  0.383720
4  2.163907 -0.164687
5  0.481719 -0.201422


Here is a summary:

In [24]:
dfPoints.describe()

,X,Y
count,30.000000,30.000000
mean,0.091400,-0.246895
std,1.144204,0.862869
min,-2.091152,-2.208408
25%,-0.674341,-0.669343
50%,0.049510,-0.163167
75%,0.996438,0.341158
max,2.163907,1.417509


Here is a plot of the points:

In [25]:
fig = px.scatter(dfPoints, x="X", y="Y", template="plotly_dark")
fig.show()

A typical pipeline of geometric nearest neighbors processing:

In [26]:
gnnObj = (GeometricNearestNeighborsProcessor(dfPoints)
   .make_nearest_function(distance_function = "EuclideanDistance")
   .compute_thresholds(number_of_nearest_neighbors = 10, aggregation_function = "mean", outlier_identifier = "QuartileIdentifierParameters")
   .find_anomalies()
   .echo_function_value("Anomaly points:")
)

Anomaly points:
          X         Y    Radius
0 -1.567601 -2.208408  1.726678
1 -1.939914  1.060095  1.352711


One way to plot the anomalies together with the data points using the package "plotly" is to merge into one data frame and add an indicator column:

In [27]:
# Assign data
df1 = gnnObj.take_data()
df2 = gnnObj.take_value()

# Mark points that are in df2
df1['highlight'] = df1.merge(df2.assign(flag=1), on=['X', 'Y'], how='left')['flag'].fillna(0)

# Convert to category for plotting
df1['highlight'] = df1['highlight'].map({0: 'data', 1: 'anomalies'})

# Plot
fig = px.scatter(df1, x='X', y='Y', color='highlight',
                 color_discrete_map={'data': 'blue', 'anomalies': 'red'},
                 template="plotly_dark")

fig.show()

Alternatively, the anomaly points can be shown as smaller, in red, and on top of the data points:

In [28]:
fig = go.Figure()

# All points (larger)
fig.add_trace(go.Scatter(
    x=df1['X'],
    y=df1['Y'],
    mode='markers',
    marker=dict(size=12, color='blue'),
    name='data'
))

# Anomaly points (smaller, on top)
fig.add_trace(go.Scatter(
    x=df2['X'],
    y=df2['Y'],
    mode='markers',
    marker=dict(size=6, color='red', line=dict(width=1, color='black')),
    name='anomalies'
))

fig.update_layout(
    template="plotly_dark",
    xaxis_title="X Axis",
    yaxis_title="Y Axis"
)

fig.show()

----

## Classification

Here we generate another set of random points using the same random point generators:

In [29]:
dfPoints2 = random_data_frame(n_rows=40, columns_spec = ["X", "Y"], generators= {"X": numpy.random.normal, "Y": numpy.random.normal})
print(dfPoints2.shape)

(40, 2)


Here the points of second set are classified into being anomalous or not:

In [30]:
gnnObj.classify(dfPoints2).take_value()

,Index,Radius,Label
0,1,1.035062,True
1,2,1.448526,False
2,3,0.870763,True
3,4,0.586735,True
4,5,1.059727,True
5,6,0.539341,True
6,7,0.788486,True
7,8,0.529060,True
8,9,0.814606,True
9,10,1.316992,False


Plot the original data points (in gray) and the new data points by marking the anomalous ones:

In [31]:
df0 = gnnObj.take_data()

dfClassified = gnnObj.classify(dfPoints2).take_value()
dfCombined = dfPoints2.join(dfClassified)

df1 = dfCombined[dfCombined["Label"] == True]
df2 = dfCombined[dfCombined["Label"] == False]

fig = go.Figure()

# Original data points
fig.add_trace(go.Scatter(
    x=df0['X'],
    y=df0['Y'],
    mode='markers',
    marker=dict(size=8, color='gray'),
    name='data'
))

# Non-anomaly points
fig.add_trace(go.Scatter(
    x=df1['X'],
    y=df1['Y'],
    mode='markers',
    marker=dict(size=10, color='blue', line=dict(width=1, color='black')),
    name='non-anomalies'
))

# Anomaly points
fig.add_trace(go.Scatter(
    x=df2['X'],
    y=df2['Y'],
    mode='markers',
    marker=dict(size=10, color='red', line=dict(width=1, color='black')),
    name='anomalies'
))

fig.update_layout(
    template="plotly_dark",
    xaxis_title="X Axis",
    yaxis_title="Y Axis"
)

fig.show()

----

## Distance matrix

Here is the distance matrix:

In [39]:
mat=gnnObj.compute_proximity_matrix().take_value()
mat

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 330 stored elements and shape (30, 30)>

Here is a corresponding matrix plot:

In [46]:
rows, cols = mat.nonzero()

fig = go.Figure(data=go.Scattergl(
    x=cols,
    y=rows,
    mode='markers',
    marker=dict(size=3)
))

fig.update_layout(
    title="Sparse Matrix Visualization",
    xaxis_title="Columns",
    yaxis_title="Rows",
    yaxis_autorange='reversed',
    xaxis=dict(scaleanchor='y', scaleratio=1),
    yaxis=dict(constrain='domain'),
    width=600,
    height=600,
    template="plotly_dark"
)

fig.show()

---

## References

[AAp1] Anton Antonov, [GeometricNearestNeighborsProcessor](https://github.com/antononcube/Python-GeometricNearestNeighborsProcessor), Python package, (2026), [GitHub/antononcube](https://github.com/antononcube).([PIPy.org](https://pypi.org/project/GeometricNearestNeighborsProcessor/).)

[AAp2] Anton Antonov, [RandomDataGenerators](https://github.com/antononcube/Python-packages/tree/main/RandomDataGenerators), Python package, (2021-2026), [GitHub/antononcube](https://github.com/antononcube). ([PIPy.org](https://pypi.org/project/RandomDataGenerators/).)

[AAp3] Anton Antonov, [OutlierIdentifiers](https://github.com/antononcube/Python-packages/tree/main/OutlierIdentifiers), Python package, (2024), [GitHub/antononcube](https://github.com/antononcube). ([PIPy.org](https://pypi.org/project/OutlierIdentifiers/).)